In [ ]:
from pathlib import Path
import tarfile
from tqdm import tqdm
from huggingface_hub import hf_hub_download
ROOT = Path.cwd()

In [ ]:
OUT_DIR = ROOT / "data" / "agvision"
EXTRACT = True
REPO = "shi-labs/Agriculture-Vision"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)

In [ ]:
print(f"Downloading {REPO} (tar.gz, ~21 GB)...")

archive = hf_hub_download(
    repo_id=REPO,
    filename="Agriculture-Vision-2021.tar.gz",
    repo_type="dataset",
    local_dir=str(OUT_DIR),
)

archive_path = Path(archive)

print("Saved:", archive_path)


In [ ]:
import tarfile
from pathlib import Path
from tqdm import tqdm  


if EXTRACT:
    print("Extracting...")
    OUT_DIR = Path(OUT_DIR)

    with tarfile.open(archive_path, "r:gz") as tf:
        members = tf.getmembers()
        
        with tqdm(total=len(members), desc="Extracting", unit="files") as pbar:
            def track_progress(tar):
                for member in tar:
                    pbar.update(1)
                    yield member

            import sys
            if sys.version_info >= (3, 12):
                tf.extractall(path=OUT_DIR, members=track_progress(tf), filter='data')
            else:
                tf.extractall(path=OUT_DIR, members=track_progress(tf))

    print(f"Extracted to {OUT_DIR}")

    expected = (
        OUT_DIR
        / "Agriculture-Vision-2021"
        / "train"
        / "images"
        / "rgb"
    )

    if expected.is_dir():
        print("OK:", expected)
    else:
        print(
            "Warning: expected "
            "Agriculture-Vision-2021/train/images/rgb "
            "— check archive layout."
        )

In [9]:
SOURCE_DIR = OUT_DIR / "Agriculture-Vision-2021"
ARCHIVE_NAME = OUT_DIR / "Agriculture-Vision-2021-processed.tar.gz"

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Directory not found: {SOURCE_DIR}")

print(f"Packing {SOURCE_DIR}...")
print(f"Output archive: {ARCHIVE_NAME}")

files = list(SOURCE_DIR.rglob("*"))

with tarfile.open(ARCHIVE_NAME, "w:gz") as tf:
    for path in tqdm(
        files,
        desc="Packing",
        unit="files"
    ):
        tf.add(
            path,
            arcname=path.relative_to(SOURCE_DIR.parent)
        )

print("Done.")
print(f"Archive saved to: {ARCHIVE_NAME}")

KeyboardInterrupt: 